In [ ]:
using Random
using Statistics
using Plots
using DataStructures
using JLD2


function neighbors(N::Int)
    nbrs = [Tuple{Int, Int}[] for _ in 1:N, _ in 1:N]
    for i in 1:N
        for j in 1:N
            if i > 1
                push!(nbrs[i, j], (i - 1, j))
            end
            if i < N
                push!(nbrs[i, j], (i + 1, j))
            end
            if j > 1
                push!(nbrs[i, j], (i, j - 1))
            end
            if j < N
                push!(nbrs[i, j], (i, j + 1))
            end
        end
    end
    return nbrs
end


function deg(N::Int)
    degs = zeros(Int, N, N)
    for i in 1:N
        for j in 1:N
            degs[i, j] = 4 - (i == 1) - (i == N) - (j == 1) - (j == N)
        end
    end
    return degs
end


function reaction_pairs(N::Int, h::Float64, ϵ::Float64, λ::Float64)
    react_pairs = Tuple{Tuple{Int,Int},Tuple{Int,Int},Float64}[]

    reaction_A = [Int[] for i in 1:N, j in 1:N]
    reaction_B = [Int[] for i in 1:N, j in 1:N]

    ncells = ceil(Int, ϵ/h)

    for i in 1:N
        for j in 1:N
            center_x_A = (i - 0.5) * h
            center_y_A = (j - 0.5) * h

            x_B_min = max(1, i - ncells)
            x_B_max = min(N, i + ncells)
            y_B_min = max(1, j - ncells)
            y_B_max = min(N, j + ncells)

            for x_B in x_B_min:x_B_max
                for y_B in y_B_min:y_B_max
                    center_x_B = (x_B - 0.5) * h
                    center_y_B = (y_B - 0.5) * h

                    r2 = (center_x_A - center_x_B)^2 + (center_y_A - center_y_B)^2

                    if r2 <= ϵ^2
                        push!(react_pairs, ((i, j), (x_B, y_B), λ))
                        p = length(react_pairs)
                        push!(reaction_A[i, j], p)
                        push!(reaction_B[x_B, y_B], p)
                    end
                end
            end
        end
    end

    return react_pairs, reaction_A, reaction_B

end



struct Channel_Indexes
    A_diff_channel:: Matrix{Int}
    B_diff_channel:: Matrix{Int}
    react_channel:: Vector{Int}
    total_channels:: Int
end

function build_channels(N::Int, react_pairs)

    N_voxels = N * N
    N_reactions = length(react_pairs)

    A_diff_channel = zeros(Int, N, N)
    B_diff_channel = zeros(Int, N, N)
    react_channel = zeros(Int, N_reactions)

    idx = 1
    for i in 1:N
        for j in 1:N
            A_diff_channel[i, j] = idx
            idx += 1
        end
    end

    for i in 1:N
        for j in 1:N
            B_diff_channel[i, j] = idx
            idx += 1
        end
    end

    for pairs in 1:N_reactions
        react_channel[pairs] = idx
        idx += 1
    end

    total_channels = 2 * N_voxels + N_reactions

    return Channel_Indexes(A_diff_channel, B_diff_channel, react_channel, total_channels)

end

function build_dependency_graph(channels, react_pairs, reaction_A, reaction_B, nrbs)

    N = size(channels.A_diff_channel, 1)
    total_channels = channels.total_channels

    dependency_graph = [Int[] for i in 1:total_channels]

    #A-Diffusion dependencies
    for i in 1:N
        for j in 1:N

            Achannel = channels.A_diff_channel[i, j]
            dependency_channels = Set{Int}()

            # Its own channel
            push!(dependency_channels, Achannel)

            # Neighbors
            for (x, y) in nrbs[i, j]
                push!(dependency_channels, channels.A_diff_channel[x, y])
            end

            # Reactions involving A in (i, j)
            for rA in reaction_A[i, j]
                push!(dependency_channels, channels.react_channel[rA])
            end

            # Reactions involving A at neighbors
            for (x, y) in nrbs[i, j]
                for rA in reaction_A[x, y]
                    push!(dependency_channels, channels.react_channel[rA])
                end
            end

            dependency_graph[Achannel] = collect(dependency_channels)
        end
    end

    #B-Diffusion dependencies
    for i in 1:N
        for j in 1:N
            Bchannel = channels.B_diff_channel[i, j]
            dependency_channels = Set{Int}()

            # Its own channel
            push!(dependency_channels, Bchannel)

            # Neighbors
            for (x, y) in nrbs[i, j]
                push!(dependency_channels, channels.B_diff_channel[x, y])
            end

            # Reactions involving B in (i, j)
            for rB in reaction_B[i, j]
                push!(dependency_channels, channels.react_channel[rB])
            end

            # Reactions involving B at neighbors
            for (x, y) in nrbs[i, j]
                for rB in reaction_B[x, y]
                    push!(dependency_channels, channels.react_channel[rB])
                end
            end

            dependency_graph[Bchannel] = collect(dependency_channels)
        end
    end

    #Reaction dependencies
    for p in 1:length(react_pairs)
        (xA, yA), (xB, yB), _ = react_pairs[p]
        react_channel = channels.react_channel[p]
        dependency_channels = Set{Int}()

        # Its own channel
        push!(dependency_channels, react_channel)

        # Diffusion channels of reactants
        push!(dependency_channels, channels.A_diff_channel[xA, yA])
        push!(dependency_channels, channels.B_diff_channel[xB, yB])

        # Reactions involving A in (xA, yA)
        for rA in reaction_A[xA, yA]
            push!(dependency_channels, channels.react_channel[rA])
        end

        # Reactions involving B in (xB, yB)
        for rB in reaction_B[xB, yB]
            push!(dependency_channels, channels.react_channel[rB])
        end

        dependency_graph[react_channel] = collect(dependency_channels)
    end

    return dependency_graph

end

function propensities(nA, nB, channels::Channel_Indexes, deg, κA, κB, react_pairs)

    propensities = zeros(Float64, channels.total_channels)

    N = size(nA, 1)

    #A-Diffusion
    
    for i in 1:N
        for j in 1:N
            Achannel = channels.A_diff_channel[i, j]
            propensities[Achannel] = κA * deg[i, j] * nA[i, j]
        end
    end

    #B-Diffusion
    for i in 1:N
        for j in 1:N
            Bchannel = channels.B_diff_channel[i, j]
            propensities[Bchannel] = κB * deg[i, j] * nB[i, j]
        end
    end

    #Reactions
    for p in 1:length(react_pairs)
        (xA, yA), (xB, yB), λxy = react_pairs[p]
        react_channel = channels.react_channel[p]
        propensities[react_channel] = λxy * nA[xA, yA] * nB[xB, yB]
    end

    return propensities

end

function update_propensities(ch::Int, nA, nB, channels::Channel_Indexes, degs, κA, κB, react_pairs, N::Int, N_voxels::Int)

    #A-Diffusion
    if ch <= N_voxels
        i = div(ch - 1, N) + 1
        j = mod(ch - 1, N) + 1
        return κA * degs[i, j] * nA[i, j]
    end

    #B-Diffusion
    if ch > N_voxels && ch <= 2 * N_voxels
        i = div(ch - N_voxels - 1, N) + 1
        j = mod(ch - N_voxels - 1, N) + 1
        return κB * degs[i, j] * nB[i, j]
    end

    #Reactions
    if ch > 2 * N_voxels
        p = ch - 2 * N_voxels
        (xA, yA), (xB, yB), λxy = react_pairs[p]
        return λxy * nA[xA, yA] * nB[xB, yB]
    end

end



 
function priorityqueue_jump_time(t, channels::Channel_Indexes, propensities, rng::AbstractRNG) #heaps can be used instead of priority queue for better performance
    total_channels = channels.total_channels

    T = zeros(Float64, total_channels)
    priorityqueue = PriorityQueue{Int, Float64}()

    for i in 1:total_channels
        if propensities[i] > 0
            T[i] = t + randexp(rng)/propensities[i]
        else
            T[i] = Inf
        end
        priorityqueue[i] = T[i]
    end
    return T, priorityqueue
end

struct CRDMEsetup
    N::Int
    N_voxels::Int
    κA::Float64
    κB::Float64
    degs::Matrix{Int}
    nrbs
    react_pairs
    reaction_A
    reaction_B
    channels::Channel_Indexes
    dependency_graph::Vector{Vector{Int}}
end

function setup_crdme(L::Float64, h::Float64, DA::Float64, DB::Float64, λ::Float64, ε::Float64)

    N = Int(ceil(L / h))
    N_voxels = N * N
    κA = DA / (h * h)
    κB = DB / (h * h)

    degs = deg(N)
    nrbs = neighbors(N)
    react_pairs, reaction_A, reaction_B = reaction_pairs(N, h, ε, λ)
    channels = build_channels(N, react_pairs)
    dependency_graph = build_dependency_graph(channels, react_pairs, reaction_A, reaction_B, nrbs)

    return CRDMEsetup(
        N, N_voxels, κA, κB,
        degs, nrbs, react_pairs, reaction_A, reaction_B,
        channels, dependency_graph
    )

end




function crdme_ssa_multiple_particles_2D_reflecting( rng::AbstractRNG,      setup::CRDMEsetup;
    NA0::Int,
    NB0::Int,
    Tfinal::Float64 = Inf)


    #Initilization
    N = setup.N
    N_voxels = setup.N_voxels
    κA = setup.κA
    κB = setup.κB
    degs = setup.degs
    nrbs = setup.nrbs
    react_pairs = setup.react_pairs
    channels = setup.channels
    dependency_graph = setup.dependency_graph

    nA = zeros(Int, N, N)
    nB = zeros(Int, N, N)

    for k in 1:NA0
        i = rand(rng, 1:N)
        j = rand(rng, 1:N)
        nA[i, j] += 1
    end

    for k in 1:NB0
        i = rand(rng, 1:N)
        j = rand(rng, 1:N)
        nB[i, j] += 1
    end


    props = propensities(nA, nB, channels, degs, κA, κB, react_pairs)

    t = 0.0

    T, priorityqueue = priorityqueue_jump_time(t, channels, props, rng)

    total_A = sum(nA)
    total_B = sum(nB)


    while t < Tfinal && total_A > 0 && total_B > 0

        (channel, t_next) = peek(priorityqueue)

        if t_next>= Tfinal
            break
        end

        t = t_next


        #A-Diffusion
        if channel <= N_voxels
            i = div(channel - 1, N) + 1
            j = mod(channel - 1, N) + 1
            nbrs = nrbs[i, j]
            if !isempty(nbrs)
                (x, y) = rand(rng, nbrs)
                if nA[i, j] > 0
                    nA[i, j] -= 1
                    nA[x, y] += 1
                end
            end
        end

        #B-Diffusion
        if channel > N_voxels && channel <= 2 * N_voxels
            i = div(channel - N_voxels - 1, N) + 1
            j = mod(channel - N_voxels - 1, N) + 1
            nbrs = nrbs[i, j]
            if !isempty(nbrs)
                (x, y) = rand(rng, nbrs)
                if nB[i, j] > 0
                    nB[i, j] -= 1
                    nB[x, y] += 1
                end
            end
        end

        #Reactions
        if channel > 2 * N_voxels
            p = channel - 2 * N_voxels
            (xA, yA), (xB, yB), _ = react_pairs[p]
            if nA[xA, yA] > 0 && nB[xB, yB] > 0
                nA[xA, yA] -= 1
                nB[xB, yB] -= 1
                total_A -= 1
                total_B -= 1
            end
        end


        #Update propensities and jump times for affected channels
        for ch in dependency_graph[channel]

            old_propensity = props[ch]
            new_propensity = update_propensities(ch, nA, nB, channels, degs, κA, κB, react_pairs, N, N_voxels)
            props[ch] = new_propensity

            if ch == channel
                T[ch] = t + randexp(rng)/new_propensity
            else
                if old_propensity >0 && new_propensity > 0
                T[ch] = t + old_propensity * (T[ch] - t) / new_propensity
                elseif new_propensity > 0
                    T[ch] = t + randexp(rng)/new_propensity
                else
                    T[ch] = Inf
                end
            end
            priorityqueue[ch] = T[ch]


        end


    end

    return t, nA, nB
end

function run_multiple_simulations(
    num_sims::Int;
    setup::CRDMEsetup,
    NA0::Int,
    NB0::Int,
    Tfinal::Float64 = Inf,
    init_same_voxel_ok::Bool = true
)
    Ts = zeros(Float64, num_sims)

    for sim in 1:num_sims
        rng = Xoshiro(sim)
        t, _, _ = crdme_ssa_multiple_particles_2D_reflecting(
            rng,
            setup;
            NA0=NA0, NB0=NB0,
            Tfinal=Tfinal
        )
        Ts[sim] = t
    end

    return Ts
end



function survival_curve(Ts::Vector{Float64}, tgrid::AbstractVector{Float64})

    Ts_sorted = sort(Ts)
    n = length(Ts_sorted)
    S = Vector{Float64}(undef, length(tgrid))

    j = 1
    for (i, t) in enumerate(tgrid)
        while j <= n && Ts_sorted[j] <= t
            j += 1
        end
        S[i] = (n - j + 1) / n
    end

    return max.(S, 1e-12)
end

function main()
    L = 0.2
    NA0 = 10
    NB0 = 10
    DA = 10.0
    DB = 10.0
    λ = 1e9
    ε = 1e-3
    Tfinal = 0.03
    num_sims = 10_000

    hs = [1e-3 / 0.02, 1e-3 / 0.04, 1e-3 / 0.08, 1e-3 / 0.16, 1e-3 / 0.32]

    tgrid = range(0, Tfinal, length=1000)

    plt = plot(
        title = "Survival Curves for Multiple Particles",
        xlabel = "t (s)",
        ylabel = "Pr[T > t]",
        xlim = (0, Tfinal),
        ylim = (1e-3, 1),
        yscale = :log10,
        legend = :topright
    )

    for h in hs
        setup = setup_crdme(L, h, DA, DB, λ, ε)
        Ts = run_multiple_simulations(
            num_sims, setup=setup, NA0=NA0, NB0=NB0, Tfinal=Tfinal
        )
        @save "Ts_h=$(h).jld2" Ts
    end

    for h in hs
        @load "Ts_h=$(h).jld2" Ts
        S = survival_curve(Ts, tgrid)
        plot!(plt, tgrid, S, label="h=$(h)")
    end

    display(plt)
end

main() #








                  
            

